In [2]:
from Bio.PDB import MMCIFParser, PDBParser
import pandas as pd
import warnings
import requests
import pickle
import torch
import time
import os
import re
import gc
from tqdm import tqdm

gc.enable()
warnings.filterwarnings("ignore", module="Bio")

In [ ]:
BIOLIP_FILE = "./BioLiP.txt"
PDB_FASTA_API = "https://www.rcsb.org/fasta/entry/{pdb_id}/display"

BIOLIP_HEADER = [
    "pdb_id",
    "receptor_chain",
    "resolution",
    "binding_site",
    "ligand_ccd_id",
    "ligand_chain",
    "ligand_serial_num",
    "binding_site_pdb", # pocket
    "binding_site_reorder",
    "catalyst_site_pdb",
    "catalyst_site_reorder",
    "enzyme_class_id",
    "go_term_id",
    "binding_affinity_literature",
    "binding_affinity_binding_moad",
    "binding_affinity_pdbind_cn",
    "binding_affinity_binding_db",
    "uniprot_db",
    "pubmed_id",
    "ligand_res_num",
    "receptor_seq"
]

In [3]:
complexes = pd.read_csv(BIOLIP_FILE, sep="\t", names=BIOLIP_HEADER)
complexes.drop_duplicates(subset="pdb_id", inplace=True)
complexes = complexes.loc[complexes.resolution<5]
complexes.reset_index(drop=True, inplace=True)
complexes

/tmp/ipykernel_1828884/2240860691.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  complexes = pd.read_csv(BIOLIP_FILE, sep="\t", names=BIOLIP_HEADER)


,pdb_id,receptor_chain,resolution,binding_site,ligand_ccd_id,ligand_chain,ligand_serial_num,binding_site_pdb,binding_site_reorder,catalyst_site_pdb,...,enzyme_class_id,go_term_id,binding_affinity_literature,binding_affinity_binding_moad,binding_affinity_pdbind_cn,binding_affinity_binding_db,uniprot_db,pubmed_id,ligand_res_num,receptor_seq
0,148l,E,1.90,BS01,peptide,S,0,Q105 M106 F114 N116 S117 K135 S136 R137 W138 Q...,Q105 M106 F114 N116 S117 K135 S136 R137 W138 Q...,NaN,...,3.2.1.17,"0003796,0003824,0008152,0009253,0016787,001679...",NaN,NaN,NaN,NaN,P00720,8266098.0,166 ~ 170,MNIFEMLRIDEGLRLKIYKDTEGYYEIGIGHLLTKSPSLNAAKSEL...
1,1a07,A,2.20,BS01,peptide,C,0,R158 H204 Y205 K206,R14 H60 Y61 K62,NaN,...,2.7.10.2,NaN,NaN,NaN,NaN,NaN,P12931,9174343.0,101 ~ 103,SIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYCLS...
2,1a08,A,2.20,BS01,peptide,C,0,R158 R178 S180 C188 K203 H204 Y205 K206,R14 R34 S36 C44 K59 H60 Y61 K62,NaN,...,2.7.10.2,NaN,NaN,NaN,NaN,NaN,P12931,9174343.0,101 ~ 103,SIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYCLS...
3,1a09,A,2.00,BS01,peptide,C,0,R158 R178 S180 E181 T182 Y187 C188 K203 H204 Y...,R15 R35 S37 E38 T39 Y44 C45 K60 H61 Y62 K63 G96,NaN,...,2.7.10.2,NaN,NaN,NaN,NaN,NaN,P12931,9174343.0,101 ~ 103,DSIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYCL...
4,1a0n,B,-1.00,BS01,peptide,A,0,Y101 Y103 W129 P144 N146 Y147,Y8 Y10 W36 P51 N53 Y54,NaN,...,2.7.10.2,NaN,NaN,NaN,NaN,NaN,P06241,8961927.0,91 ~ 104,VTLFVALYDYEARTEDDLSFHKGEKFQILNSSEGDWWEARSLTTGE...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14743,8wu8,A,2.81,BS01,peptide,D,0,R45 S46 A47 Y48 Q133 A134 V135 E165 A263 T264 ...,R45 S46 A47 Y48 Q127 A128 V129 E159 A252 T253 ...,NaN,...,3.1.11.2,NaN,NaN,NaN,NaN,NaN,Q99638,NaN,88 ~ 98,MKCLVTGGNVKVLGKAVHSLSRIGDELYLEPLEDGLSLRTVNSSRS...
14744,8wx5,A,3.91,BS01,peptide,B,0,R48 Y52 Y86 K172 A180 F195 N196 Y199 Y335 E465...,R21 Y25 Y59 K145 A153 F168 N169 Y172 Y268 E398...,NaN,...,?,"0002376,0005290,0005515,0005764,0005765,000576...",NaN,NaN,NaN,NaN,Q8N697,NaN,1 ~ 14,GAFAGRRAACGAVLLTELLERAAFYGITSNLVLFLNGAPFCWEGAQ...
14745,8xgc,I,3.70,BS01,peptide,K,0,K104 K107 R181 L184 K261 F265 T268 S271 N443 S447,K94 K97 R171 L174 K251 F255 T258 S261 N410 S414,NaN,...,?,NaN,NaN,NaN,NaN,NaN,P53840,NaN,324 ~ 343,NAADFSLTVLRARIALLATAIGGPDYTSQIDPPPYKLGDDCLACLK...
14746,9ins,B,1.70,BS01,peptide,A,0,V2 N3 Q4 H5 L6 C7 L15 C19 R22 G23 F24 F25,V2 N3 Q4 H5 L6 C7 L15 C19 R22 G23 F24 F25,NaN,...,?,"0005179,0005576",NaN,NaN,NaN,NaN,P01315,1504238.0,1 ~ 21,FVNQHLCGSHLVEALYLVCGERGFFYTPKA


In [ ]:
def get_chain_auth_ids(pdb_seq_name):
    chain_ids = pdb_seq_name.split("|")[1]
    chain_ids = chain_ids[6:].split(",")
    pattern = re.compile(r'\s*(.*?)\[auth (.*?)\]')
    all_chain_auth_ids = []
    for c in chain_ids:
        if("auth" in c):
            match = pattern.match(c)
            if match:
                # chain_id = match.group(1).strip()
                chain_auth_id = match.group(2).strip()
        else:
            chain_auth_id = c.strip()
        all_chain_auth_ids.append(chain_auth_id)
    return all_chain_auth_ids

def get_sequence_from_pdb_fasta(pdb_id):
    r = requests.get(PDB_FASTA_API.format(pdb_id=pdb_id))
    lines = r.text.split("\n")
    seqs = {}

    for l in lines:
        if(len(l)==0):
            break
        if(l[0] == ">"):# sequence name
            chain_ids = get_chain_auth_ids(l)
        else:# actual sequence
            assert len(chain_ids)!=0 # double check
            for c in chain_ids:
                seqs[c] = l.strip()
    return seqs

pdb_id = "8snb"
ligand_chain = "1r"
receptor_chain = "1y"

seqs = get_sequence_from_pdb_fasta(pdb_id)
ligand_seq = seqs[ligand_chain]
receptor_seq = seqs[receptor_chain]
ligand_seq, receptor_seq

In [ ]:
all_ligand_seqs = []

for _, r in tqdm(complexes.iterrows(), total=complexes.shape[0]):
    seqs = get_sequence_from_pdb_fasta(r.pdb_id)

    if(r.ligand_chain in seqs.keys()):
        all_ligand_seqs.append((r.pdb_id, seqs[r.ligand_chain]))
    else:
        all_ligand_seqs.append((r.pdb_id, False))

    time.sleep(1)

In [ ]:
with open("./all_ligand_seq.pkl", "+wb") as f:
    pickle.dump(all_ligand_seqs, f)
len(all_ligand_seqs)

In [ ]:
ALL_LIGAND_SEQ_FILE = "./all_ligand_seq.pkl"

with open(ALL_LIGAND_SEQ_FILE, "rb") as f:
    all_ligand_seq = pickle.load(f)

complexes["ligand_seq"] = [l[1] for l in all_ligand_seq]
complexes

,pdb_id,receptor_chain,resolution,binding_site,ligand_ccd_id,ligand_chain,ligand_serial_num,binding_site_pdb,binding_site_reorder,catalyst_site_pdb,...,go_term_id,binding_affinity_literature,binding_affinity_binding_moad,binding_affinity_pdbind_cn,binding_affinity_binding_db,uniprot_db,pubmed_id,ligand_res_num,receptor_seq,ligand_seq
0,148l,E,1.90,BS01,peptide,S,0,Q105 M106 F114 N116 S117 K135 S136 R137 W138 Q...,Q105 M106 F114 N116 S117 K135 S136 R137 W138 Q...,NaN,...,"0003796,0003824,0008152,0009253,0016787,001679...",NaN,NaN,NaN,NaN,P00720,8266098.0,166 ~ 170,MNIFEMLRIDEGLRLKIYKDTEGYYEIGIGHLLTKSPSLNAAKSEL...,AEKA
1,1a07,A,2.20,BS01,peptide,C,0,R158 H204 Y205 K206,R14 H60 Y61 K62,NaN,...,NaN,NaN,NaN,NaN,NaN,P12931,9174343.0,101 ~ 103,SIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYCLS...,XYEX
2,1a08,A,2.20,BS01,peptide,C,0,R158 R178 S180 C188 K203 H204 Y205 K206,R14 R34 S36 C44 K59 H60 Y61 K62,NaN,...,NaN,NaN,NaN,NaN,NaN,P12931,9174343.0,101 ~ 103,SIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYCLS...,XYEX
3,1a09,A,2.00,BS01,peptide,C,0,R158 R178 S180 E181 T182 Y187 C188 K203 H204 Y...,R15 R35 S37 E38 T39 Y44 C45 K60 H61 Y62 K63 G96,NaN,...,NaN,NaN,NaN,NaN,NaN,P12931,9174343.0,101 ~ 103,DSIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYCL...,XYEX
4,1a0n,B,-1.00,BS01,peptide,A,0,Y101 Y103 W129 P144 N146 Y147,Y8 Y10 W36 P51 N53 Y54,NaN,...,NaN,NaN,NaN,NaN,NaN,P06241,8961927.0,91 ~ 104,VTLFVALYDYEARTEDDLSFHKGEKFQILNSSEGDWWEARSLTTGE...,PPRPLPVAPGSSKT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14743,8wu8,A,2.81,BS01,peptide,D,0,R45 S46 A47 Y48 Q133 A134 V135 E165 A263 T264 ...,R45 S46 A47 Y48 Q127 A128 V129 E159 A252 T253 ...,NaN,...,NaN,NaN,NaN,NaN,NaN,Q99638,NaN,88 ~ 98,MKCLVTGGNVKVLGKAVHSLSRIGDELYLEPLEDGLSLRTVNSSRS...,TSKFPHLTFESP
14744,8wx5,A,3.91,BS01,peptide,B,0,R48 Y52 Y86 K172 A180 F195 N196 Y199 Y335 E465...,R21 Y25 Y59 K145 A153 F168 N169 Y172 Y268 E398...,NaN,...,"0002376,0005290,0005515,0005764,0005765,000576...",NaN,NaN,NaN,NaN,Q8N697,NaN,1 ~ 14,GAFAGRRAACGAVLLTELLERAAFYGITSNLVLFLNGAPFCWEGAQ...,MLSEGYLSGLEYWND
14745,8xgc,I,3.70,BS01,peptide,K,0,K104 K107 R181 L184 K261 F265 T268 S271 N443 S447,K94 K97 R171 L174 K251 F255 T258 S261 N410 S414,NaN,...,NaN,NaN,NaN,NaN,NaN,P53840,NaN,324 ~ 343,NAADFSLTVLRARIALLATAIGGPDYTSQIDPPPYKLGDDCLACLK...,MDDALHALSSLTAKKRTTTYKKVAVPILDENDNTNGNGPNDIDNPP...
14746,9ins,B,1.70,BS01,peptide,A,0,V2 N3 Q4 H5 L6 C7 L15 C19 R22 G23 F24 F25,V2 N3 Q4 H5 L6 C7 L15 C19 R22 G23 F24 F25,NaN,...,"0005179,0005576",NaN,NaN,NaN,NaN,P01315,1504238.0,1 ~ 21,FVNQHLCGSHLVEALYLVCGERGFFYTPKA,GIVEQCCTSICSLYQLENYCN


In [12]:
# Remove 'X'
temp = [i for i in all_ligand_seq if i[1]!=False]
all_ligand_seq = [i for i in temp if ("X" not in i[1])]
len(all_ligand_seq)

11984

In [14]:
# trim by length
all_ligand_seq = [i for i in all_ligand_seq if (len(i[1])>=5 and len(i[1])<=32)]
len(all_ligand_seq)

9875

In [ ]:
# split into train & val & test
all_fasta = ""
for s in all_ligand_seq:
    if(s[1]!=False):
        all_fasta += f">{s[0]}\n{s[2]}\n"
with open("all_receptors_no_x_trim_length.fasta", "+w") as f:
    f.write(all_fasta)
# mmseqs easy-cluster all_receptors_no_x_trim_length.fasta all_receptors_no_x_trim_length tmp --min-seq-id 0.5 -c 0.8 --cov-mode 1 --threads 64

In [68]:
cluster_res = pd.read_csv("./all_receptors_no_x_trim_length_cluster.tsv", sep="\t", header=None)
cluster_res.columns = ["cluster", "id"]
cluster_res = cluster_res.groupby("cluster").apply(lambda df: df.sample(len(df)) if len(df)<10 else df.sample(10, random_state=0)).reset_index(drop=True)
group_count = cluster_res.groupby("cluster").count()

possible_testing_set = group_count.sample(n=int(len(cluster_res)*0.05), random_state=0)
possible_testing_ids = possible_testing_set.index
rest_set = group_count.loc[~group_count.index.isin(possible_testing_ids)]
rest_set = rest_set.sample(len(rest_set), random_state=0)

rest_size = rest_set.sum().id
train_size = int(rest_size*0.8)
val_size = int(rest_size*0.2)
train_size, val_size, len(possible_testing_ids)

/tmp/ipykernel_1828884/3142597114.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cluster_res = cluster_res.groupby("cluster").apply(lambda df: df.sample(len(df)) if len(df)<10 else df.sample(10, random_state=0)).reset_index(drop=True)


(3384, 846, 248)

In [69]:
train_ids = []
datapoint_counts = 0
for i, r in rest_set.iterrows():
    train_ids.append(i)
    datapoint_counts += r.id
    if(datapoint_counts>=train_size):
        break

train_set = cluster_res.loc[cluster_res.cluster.isin(train_ids)]
val_set = cluster_res.loc[~(
    cluster_res.cluster.isin(train_ids) |
    cluster_res.cluster.isin(possible_testing_set.index)
)]

In [72]:
test_ids = set(possible_testing_set.index)
train_ids = set(train_set.id)
val_ids = set(val_set.id)
print(len(test_ids) + len(train_ids) + len(val_ids), rest_set.sum().id+len(possible_testing_ids))
test_ids.intersection(train_ids), test_ids.intersection(val_ids), train_ids.intersection(val_ids)

4479 4479


(set(), set(), set())

In [ ]:
with open("./splits_no_x_trim_length.pkl", "+wb") as f:
    pickle.dump({
        "train":train_ids,
        "val":val_ids,
        "test":test_ids
    }, f)

# with open("splits_no_x.pkl", "rb") as f:
#     split_no_x = pickle.load(f)

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Model

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_uniref50')
model = T5ForConditionalGeneration.from_pretrained('Rostlab/prot_t5_xl_uniref50')
model = model.eval()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
def get_embedding(seq:str):
    #"C N C K R F P Q C P L N F L C"
    # Define your input
    sequences_Example = [" ".join(seq)]
    sequences_Example = [re.sub(r"[UZOB]", "X", sequence) for sequence in sequences_Example]
    input_seq = sequences_Example[0]

    # Tokenize the input text
    tokens = tokenizer(input_seq, add_special_tokens=True, padding=True, return_tensors="pt")
    tokens = tokens.to(device)
    with torch.no_grad():
        # Pass the input through the encoder
        encoder_outputs = model.encoder(
            input_ids=tokens.input_ids,
            attention_mask=tokens.attention_mask
        )
        # Extract the hidden states
        encoder_hidden_states = encoder_outputs.last_hidden_state.cpu()
    del sequences_Example, input_seq, tokens, encoder_outputs
    return encoder_hidden_states[0]


get_embedding("CNCKRFPQCPLNFLC").shape

In [ ]:
# Around 15~20 minutes
all_pairs = []

for i, r in tqdm(complexes.iterrows(), total=len(complexes)):
    try:
        ligand_seq = r.ligand_seq
        receptor_seq = r.receptor_seq
        ligand_emb = get_embedding(ligand_seq).numpy()
        receptor_emb = get_embedding(receptor_seq).numpy()
        receptor_contact_res = [int(i[1:]) for i in r.binding_site_reorder.split()]
        all_pairs.append((
            r.pdb_id,
            receptor_emb,
            ligand_emb,
            receptor_contact_res
        ))
        del ligand_emb, receptor_emb, receptor_contact_res
    except:
        all_pairs.append((r.pdb_id))

In [ ]:
with open("./all_emb.pkl", "+wb") as f:
    pickle.dump(all_pairs, f)

In [22]:
# all_len = ([i[1].shape[0] for i in all_pairs if type(i)==tuple])
# max(all_len), min(all_len), len(all_len)

In [23]:
# all_len = ([i[1].shape[0] for i in all_pairs if type(i)==tuple and i[1].shape[0]<=1024])
# max(all_len), min(all_len), len(all_len)

In [ ]:
# with open("./all_emb.pkl", "rb") as f:
#     all_pairs = pickle.load(f)

# Preprocessing

In [23]:
import torch
import pickle
from tqdm import tqdm
import torch.nn.functional as F
torch.set_num_threads(64)
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
all_receptor_emb = np.concatenate([i[1] for i in all_pairs if type(i)==tuple])
all_ligand_emb = np.concatenate([i[2] for i in all_pairs if type(i)==tuple])
all_emb = np.concatenate([all_receptor_emb, all_ligand_emb])
all_emb.shape

(4406774, 1024)

In [ ]:
scaler = StandardScaler()
scaler.fit(all_emb)

StandardScaler()

In [ ]:
for i in tqdm(range(len(all_pairs))):
    if(type(all_pairs[i]) == tuple):
        all_pairs[i] = [
            all_pairs[i][0],
            scaler.transform(all_pairs[i][1]), 
            scaler.transform(all_pairs[i][2]),
            all_pairs[i][3],
        ]

100%|██████████| 14748/14748 [00:19<00:00, 754.99it/s]


In [ ]:
with open("./all_emb_z_score.pkl", "+wb") as f:
    pickle.dump(all_pairs, f)

In [ ]:
with open("./z_score_scaler.pkl", "+wb") as f:
    pickle.dump(scaler, f)

In [ ]:
# with open("./all_emb_z_score.pkl", "rb") as f:
#     all_pairs = pickle.load(f)

In [ ]:
# with open("./z_score_scaler.pkl", "rb") as f:
#     scaler = pickle.load(f)

In [ ]:
max_len = 1024
pocket_ext = 1

all_pairs = [i for i in all_pairs if type(i)!=str]
all_pairs = [i for i in all_pairs if i[1].shape[0]<=max_len and i[2].shape[0]<=1024]
all_len = [i[1].shape[0] for i in all_pairs]
max(all_len), min(all_len), len(all_len)

(1024, 31, 14569)

In [ ]:
def pad(seq):
    if seq.shape[0] > max_len:
        raise RuntimeError("Length exceed:", len(seq), max_len)
    seq = F.pad(
        seq, (0, 0, 0, max_len - seq.shape[0]), mode="constant", value=0
    )
    return seq.float()

In [ ]:
transformed_data = []
for d in tqdm(all_pairs):
    receptor_emb = torch.Tensor(d[1])
    ligand_emb = torch.Tensor(d[2])
    pocket_ids = torch.Tensor(d[3])

    receptor_mask = torch.zeros(size=(max_len,))
    receptor_mask[: receptor_emb.shape[0]] = 1.0
    receptor_emb = pad(receptor_emb)

    ligand_mask = torch.zeros(size=(max_len,))
    ligand_mask[: ligand_emb.shape[0]] = 1.0
    ligand_emb = pad(ligand_emb)

    pocket_mask = torch.Tensor([(i in pocket_ids) for i in range(max_len)]).bool()
    pocket_shit_left = torch.roll(pocket_mask, pocket_ext)
    pocket_shit_left[0] = False
    pocket_shit_right = torch.roll(pocket_mask, -pocket_ext)
    pocket_shit_right[-1] = False
    pocket_mask = pocket_mask | pocket_shit_left | pocket_shit_right

    transformed_data.append({
        "receptor_emb": receptor_emb,
        "receptor_mask":receptor_mask,
        "ligand_emb": ligand_emb,
        "ligand_mask":ligand_mask,
        "pocket_mask":pocket_mask,
        "pdb_id": d[0]
    })

100%|██████████| 14569/14569 [23:09<00:00, 10.48it/s]


In [ ]:
with open("/data/bai/Drug_discovery/cleanData/BioLip/seq_emb_1024_z.pkl", "+wb") as f:
    pickle.dump(transformed_data, f)

with open("/data/bai/Drug_discovery/cleanData/BioLip/seq_emb_1024_z.pkl", "rb") as f:
    transformed_data = pickle.load(f)

In [74]:
with open("splits_no_x_trim_length.pkl", "rb") as f:
    split_no_x = pickle.load(f)

In [75]:
train_data = [i for i in transformed_data if i["pdb_id"] in split_no_x["train"]]
val_data = [i for i in transformed_data if i["pdb_id"] in split_no_x["val"]]
test_data = [i for i in transformed_data if i["pdb_id"] in split_no_x["test"]]
len(train_data), len(val_data), len(test_data), len(train_data)+len(val_data)+len(test_data)

(3341, 839, 246, 4426)

In [ ]:
with open("./seq_emb_1024_no_x_trimmed_z_train.pkl", "+wb") as f:
    pickle.dump(train_data, f)

with open("./seq_emb_1024_no_x_trimmed_z_val.pkl", "+wb") as f:
    pickle.dump(val_data, f)

with open("./seq_emb_1024_no_x_trimmed_z_test.pkl", "+wb") as f:
    pickle.dump(test_data, f)